# Day 3: Unsupervised Learning, Topic Models, and Scaling

Today is about discovering structure without document labels. The main classroom question is not just whether a model runs, but whether the output can be interpreted and validated.

By the end you should be able to:

1. Fit a simple clustering model.
2. Fit and inspect a topic model.
3. Compare topic counts.
4. Relate model output to metadata.
5. Write a short validation memo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 140)

In [ ]:
from pathlib import Path


def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


DATA_DIR = find_data_dir()
DATA_DIR

## spaCy setup

The notebooks use spaCy for tokenization and preprocessing. If `en_core_web_sm` is installed, the same setup also shows POS tags, lemmas, and dependency parses. If not, the notebooks still run with a blank English tokenizer.

In [ ]:
import spacy

try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, and dependency parses.')


def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        if keep_alpha and not token.is_alpha:
            continue
        if remove_stop and token.is_stop:
            continue
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


def spacy_analyzer(text):
    return spacy_preprocess(text)


def spacy_analyzer_with_bigrams(text):
    tokens = spacy_preprocess(text)
    bigrams = [tokens[i] + '_' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams


def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)

## 1. Clustering intuition with toy data

K-means partitions documents into clusters. Here we use two visible dimensions so the idea is clear.

In [ ]:
toy = pd.DataFrame({
    'banana': [8, 7, 6, 1, 0, 1, 5, 6],
    'chocolate': [1, 2, 1, 8, 7, 6, 5, 4]
})

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
toy['cluster'] = kmeans.fit_predict(toy[['banana', 'chocolate']])

toy

In [ ]:
plt.figure(figsize=(5, 4))
for cluster, group in toy.groupby('cluster'):
    plt.scatter(group['banana'], group['chocolate'], label=f'cluster {cluster}', s=70)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], marker='x', s=120, color='black', label='centers')
plt.xlabel('banana count')
plt.ylabel('chocolate count')
plt.legend()
plt.tight_layout()

## 2. Topic modeling on 20 Newsgroups

We use a local classroom-sized sample from 20 Newsgroups and fit LDA with scikit-learn. The known source categories are not used for training, but they give us useful validation metadata after the unsupervised model is fit.

### Methodology formulas: clustering, topics, and scaling

K-means clustering partitions documents into $K$ groups by minimizing within-cluster squared distance:

$$\min_{C_1,\ldots,C_K}\sum_{k=1}^K\sum_{i \in C_k}\lVert x_i - \mu_k\rVert_2^2.$$

Latent Dirichlet Allocation treats each document as a mixture of topics. A compact generative description is

$$\theta_d \sim \mathrm{Dirichlet}(\alpha), \qquad z_{dn} \sim \mathrm{Categorical}(\theta_d), \qquad w_{dn} \sim \mathrm{Categorical}(\phi_{z_{dn}}).$$

Here $\theta_{dk}$ is the share of topic $k$ in document $d$, and $\phi_{kv}$ is the probability of word $v$ under topic $k$:

$$P(w=v \mid d)=\sum_{k=1}^K \theta_{dk}\phi_{kv}.$$

The document scaling exercise uses a lower-dimensional representation of the document-term matrix:

$$X \approx U_k \Sigma_k V_k^\top, \qquad z_d = (U_k\Sigma_k)_{d\cdot}.$$

Interpretation should combine statistical fit with substantive validation: a topic is a useful measurement dimension only when its high-probability words, representative documents, and covariate patterns tell the same story.


In [ ]:
news = pd.read_csv(DATA_DIR / 'twenty_newsgroups_sample.csv')
news = news.dropna(subset=['text', 'category']).copy()
news['broad_area'] = np.select(
    [
        news['category'].str.startswith('talk.politics'),
        news['category'].str.startswith('sci.'),
        news['category'].str.startswith('rec.'),
        news['category'].str.startswith('comp.')
    ],
    ['politics', 'science', 'recreation', 'computing'],
    default='other'
)
news['doc_id'] = np.arange(len(news))

docs = news.reset_index(drop=True)

display(docs[['doc_id', 'category', 'broad_area', 'text']].head())
docs['category'].value_counts().sort_index()

## 2a. spaCy preprocessing for topic models

Before fitting a topic model, inspect how spaCy tokenizes a newsgroup excerpt. The topic model below uses these spaCy-preprocessed tokens.

In [ ]:
spacy_token_table(docs.loc[0, 'text'][:500]).head(25)

In [ ]:
docs['tokens'] = docs['text'].apply(spacy_analyzer)
docs[['category', 'broad_area', 'tokens']].head()

vectorizer = CountVectorizer(
    analyzer=lambda tokens: tokens,
    min_df=4,
    max_df=0.70,
    max_features=3500
)

X = vectorizer.fit_transform(docs['tokens'])
terms = vectorizer.get_feature_names_out()
X.shape

In [ ]:
n_topics = 8
lda = LatentDirichletAllocation(
    n_components=n_topics,
    learning_method='batch',
    max_iter=20,
    random_state=42
)

doc_topic = lda.fit_transform(X)
topic_word = lda.components_

In [ ]:
def top_words_for_topic(topic_index, n=12):
    weights = topic_word[topic_index]
    top_indices = weights.argsort()[::-1][:n]
    return pd.DataFrame({
        'topic': topic_index,
        'term': terms[top_indices],
        'weight': weights[top_indices]
    })

pd.concat([top_words_for_topic(k, n=10) for k in range(n_topics)], ignore_index=True)

## 2b. Topic-word weight plots

Top-word tables are easier to interpret when paired with the relative weights inside each topic.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7), sharex=False)
axes = axes.ravel()

for topic_index, ax in enumerate(axes[:n_topics]):
    topic_terms = top_words_for_topic(topic_index, n=8).sort_values('weight')
    ax.barh(topic_terms['term'], topic_terms['weight'], color='#4C72B0')
    ax.set_title(f'Topic {topic_index}')
    ax.set_xlabel('Weight')

for ax in axes[n_topics:]:
    ax.axis('off')

plt.tight_layout()

## 3. Representative documents

Top words are not enough. Representative documents help decide whether the topic label is defensible.

In [ ]:
topic_cols = [f'topic_{k}' for k in range(n_topics)]
doc_topic_df = pd.DataFrame(doc_topic, columns=topic_cols)
topic_docs = pd.concat([docs[['doc_id', 'category', 'broad_area', 'text']], doc_topic_df], axis=1)


def representative_docs(topic_index, n=5):
    col = f'topic_{topic_index}'
    return (
        topic_docs
        .sort_values(col, ascending=False)
        [['doc_id', 'category', 'broad_area', col, 'text']]
        .head(n)
    )

representative_docs(0, n=5)

Change the topic number in the next cell and inspect the documents. Do the documents support the label you would assign from top words alone?

In [ ]:
representative_docs(3, n=5)

## 4. Topic prevalence by metadata

Topic output becomes more useful when connected to document metadata. Because the model never saw the newsgroup category labels, category-topic alignment is a validation check rather than a supervised target.

In [ ]:
prevalence_by_area = topic_docs.groupby('broad_area')[topic_cols].mean().sort_index()
prevalence_by_category = topic_docs.groupby('category')[topic_cols].mean().sort_index()

prevalence_by_area

In [ ]:
prevalence_by_area.T.plot(kind='bar', figsize=(9, 4))
plt.title('Average topic prevalence by broad source area')
plt.ylabel('Mean topic proportion')
plt.tight_layout()

## 4a. Topic prevalence by source category

This view asks whether unsupervised topics align with known source categories. Strong alignment can aid interpretation, while weak or surprising alignment is a cue for closer document inspection.

In [ ]:
plt.figure(figsize=(9, 5))
sns.heatmap(prevalence_by_category, cmap='mako', cbar_kws={'label': 'Mean topic proportion'})
plt.title('Topic prevalence by newsgroup category')
plt.xlabel('Topic')
plt.ylabel('Category')
plt.tight_layout()

## 4b. Topic similarity

Topics are often not cleanly separated. A topic-similarity heatmap helps detect redundant topics.

In [ ]:
topic_similarity = cosine_similarity(topic_word)

plt.figure(figsize=(6, 5))
sns.heatmap(topic_similarity, annot=True, fmt='.2f', cmap='viridis', vmin=0, vmax=1)
plt.title('Cosine similarity between topic-word distributions')
plt.xlabel('Topic')
plt.ylabel('Topic')
plt.tight_layout()

### Additional demo: topic stability across random seeds

Topic models can look authoritative even when some topics are unstable. This cell refits the same model with several random seeds and asks how well each reference topic can be matched by top-word overlap.

In [ ]:
def topic_top_term_sets(components, n_terms=12):
    return [set(terms[row.argsort()[::-1][:n_terms]]) for row in components]

reference_sets = topic_top_term_sets(topic_word, n_terms=12)
stability_rows = []

for seed in [7, 21, 99]:
    seed_model = LatentDirichletAllocation(
        n_components=n_topics,
        learning_method='batch',
        max_iter=8,
        random_state=seed
    )
    seed_model.fit(X)
    seed_sets = topic_top_term_sets(seed_model.components_, n_terms=12)

    for ref_topic, ref_terms in enumerate(reference_sets):
        overlaps = []
        for candidate_topic, candidate_terms in enumerate(seed_sets):
            union = ref_terms | candidate_terms
            overlap = len(ref_terms & candidate_terms) / len(union)
            overlaps.append((candidate_topic, overlap))
        best_topic, best_jaccard = max(overlaps, key=lambda item: item[1])
        stability_rows.append({
            'seed': seed,
            'reference_topic': ref_topic,
            'best_matching_topic': best_topic,
            'best_jaccard': best_jaccard
        })

stability = pd.DataFrame(stability_rows)
stability_matrix = stability.pivot(index='seed', columns='reference_topic', values='best_jaccard')

plt.figure(figsize=(8, 3.5))
sns.heatmap(stability_matrix, annot=True, fmt='.2f', cmap='YlGnBu', vmin=0, vmax=1)
plt.title('Topic stability: best top-word overlap across seeds')
plt.xlabel('Reference topic from main model')
plt.ylabel('Alternative random seed')
plt.tight_layout()

stability.groupby('reference_topic')['best_jaccard'].mean().sort_values().to_frame('mean_best_jaccard')

## 5. Compare topic counts

There is no universally correct number of topics. Compare several models and inspect whether the extra topics are useful.

In [ ]:
comparison_rows = []

for k in [5, 8, 12]:
    model = LatentDirichletAllocation(n_components=k, max_iter=12, random_state=42, learning_method='batch')
    model.fit(X)
    comparison_rows.append({
        'topics': k,
        'perplexity_lower_is_better': model.perplexity(X)
    })

comparison = pd.DataFrame(comparison_rows)
comparison

## 5a. Model-comparison plot

Perplexity is not validity, but plotting it makes clear that fit diagnostics and interpretability need to be considered separately.

In [ ]:
plt.figure(figsize=(5, 4))
sns.lineplot(data=comparison, x='topics', y='perplexity_lower_is_better', marker='o')
plt.title('Topic-count comparison')
plt.ylabel('Perplexity (lower is better)')
plt.tight_layout()

Perplexity is a model-fit diagnostic, not a validity test. A lower value does not guarantee more interpretable or more useful topics.

## 6. A lightweight projection exercise

Text scaling methods are specialized models. As a simple classroom proxy, we can project TF-IDF document vectors onto one dimension and inspect whether the dimension aligns with source categories.

In [ ]:
tfidf = TfidfVectorizer(analyzer=lambda tokens: tokens, min_df=4, max_df=0.70, max_features=3500)
Z = tfidf.fit_transform(docs['tokens'])

svd = TruncatedSVD(n_components=1, random_state=42)
docs['dimension_1'] = svd.fit_transform(Z)[:, 0]

docs.groupby('category')['dimension_1'].agg(['count', 'mean', 'std']).sort_values('mean')

## 6a. Document map with dominant topics

A two-dimensional map helps detect whether topic structure tracks category, broader area, or something else.

In [ ]:
svd_map = TruncatedSVD(n_components=2, random_state=42)
map_coords = svd_map.fit_transform(Z)

docs['map_x'] = map_coords[:, 0]
docs['map_y'] = map_coords[:, 1]
docs['dominant_topic'] = doc_topic.argmax(axis=1).astype(str)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=docs,
    x='map_x',
    y='map_y',
    hue='category',
    style='dominant_topic',
    alpha=0.75,
    s=45
)
plt.title('SVD map of 20 Newsgroups posts, shaped by dominant topic')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 4.5))
sns.boxplot(data=docs, x='dimension_1', y='category', color='#4C72B0')
plt.title('One-dimensional projection by source category')
plt.xlabel('SVD dimension 1')
plt.ylabel('')
plt.tight_layout()

## 7. Validation memo template

Fill this in after inspecting at least one topic or cluster.

- Model estimated:
- Topic/cluster inspected:
- Tentative label:
- Evidence supporting the label:
- Evidence against the label:
- Possible artifact:
- Next validation check:

In [ ]:
# Your turn: choose a topic, inspect top words and representative documents, then fill in the memo above.
chosen_topic = 0
display(top_words_for_topic(chosen_topic, n=15))
representative_docs(chosen_topic, n=5)